# 🔍 Veritas — Training Pipeline

This notebook trains the text credibility layer of Veritas.
Send any URL → Veritas scrapes the content → runs all analysis layers.

| Stage | What | Dataset | Output |
|-------|------|---------|--------|
| **A** | Text credibility (DistilBERT) | LIAR (global) | `./models/liar_distilbert/` |

> **Colab users:** Set runtime to GPU → `Runtime → Change runtime type → T4 GPU`

---
## 0. Install Dependencies (Colab only)

In [ ]:
# Uncomment if running on Google Colab
!pip install torch transformers accelerate sentencepiece pillow opencv-python-headless requests beautifulsoup4 pandas scikit-learn -q

---
## 1. Imports & Configuration

In [ ]:
import os, io, zipfile, json, re, time, glob
import requests
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset as TorchDataset
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from PIL import Image

# ── Paths ────────────────────────────────────────────────
MODEL_DIR = "./models/liar_distilbert"
DATA_DIR  = "./data"
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

# ── Config ───────────────────────────────────────────────
MODEL_NAME    = "distilbert-base-uncased"
EPOCHS_GLOBAL = 3
BATCH_SIZE    = 32

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

---
# 🎤 STAGE A — Global Pre-training on LIAR

The **LIAR dataset** (12.8k labeled political statements, 6-class → collapsed to 3-class)
is downloaded directly as TSV files from the original author’s website.

Source: Wang, W.Y. (2017) *“Liar, Liar Pants on Fire”: A New Benchmark Dataset for Fake News Detection*

### A.1 Download LIAR dataset (raw TSV files)

In [ ]:

# ── LIAR dataset sources (tried in order) ─────────────────────────────────────
# 1) Original UCSB zip (may be unreachable from some networks)
# 2) GitHub mirror (reliable, individual TSV files)
# 3) Kaggle (requires kaggle.json — uncomment if the others fail)

LIAR_DIR = os.path.join(DATA_DIR, "liar")
os.makedirs(LIAR_DIR, exist_ok=True)

# The actual LIAR TSV has 14 columns (the README omits the last "context" column)
LIAR_COLUMNS = [
    "id", "label", "statement", "subject", "speaker",
    "speaker_job", "state_info", "party",
    "barely_true_count", "false_count", "half_true_count",
    "mostly_true_count", "pants_fire_count", "context",
]

# ── Source 1: UCSB zip ─────────────────────────────────────────────────────────
LIAR_UCSB_URL = "https://www.cs.ucsb.edu/~william/data/liar_dataset.zip"

# ── Source 2: GitHub mirror (individual files) ─────────────────────────────────
LIAR_GH_BASE  = "https://raw.githubusercontent.com/thiagorainmaker77/liar_dataset/master/"
LIAR_GH_FILES = {"train.tsv": f"{LIAR_GH_BASE}train.tsv",
                  "valid.tsv": f"{LIAR_GH_BASE}valid.tsv",
                  "test.tsv":  f"{LIAR_GH_BASE}test.tsv"}

HEADERS = {"User-Agent": "Mozilla/5.0 (TruthScan/1.0)"}


def _tsv_exists():
    return all(os.path.isfile(os.path.join(LIAR_DIR, f))
               for f in ("train.tsv", "valid.tsv", "test.tsv"))


def _try_ucsb():
    print("  Trying UCSB zip…")
    resp = requests.get(LIAR_UCSB_URL, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
        for member in zf.namelist():
            if member.endswith(".tsv"):
                fname = os.path.basename(member)
                dest  = os.path.join(LIAR_DIR, fname)
                with open(dest, "wb") as f:
                    f.write(zf.read(member))
                print(f"    ✅ {fname}")
    return _tsv_exists()


def _try_github():
    print("  Trying GitHub mirror…")
    ok = True
    for fname, url in LIAR_GH_FILES.items():
        dest = os.path.join(LIAR_DIR, fname)
        resp = requests.get(url, headers=HEADERS, timeout=30)
        if resp.status_code != 200:
            print(f"    ❌ {fname} — HTTP {resp.status_code}")
            ok = False
            continue
        with open(dest, "wb") as f:
            f.write(resp.content)
        print(f"    ✅ {fname}  ({len(resp.content):,} bytes)")
    return ok and _tsv_exists()


def _try_kaggle():
    """
    Kaggle fallback — requires ~/.kaggle/kaggle.json or KAGGLE_USERNAME / KAGGLE_KEY env vars.
    On Colab: upload kaggle.json first, or use: from google.colab import files; files.upload()
    Dataset slug: ucsbnlp/liar
    """
    print("  Trying Kaggle (ucsbnlp/liar)…")
    try:
        import kaggle
        kaggle.api.authenticate()
        kaggle.api.dataset_download_files("ucsbnlp/liar", path=LIAR_DIR, unzip=True)
        # Kaggle extracts as train.csv / valid.csv / test.csv — rename to .tsv
        for stem in ("train", "valid", "test"):
            csv_path = os.path.join(LIAR_DIR, f"{stem}.csv")
            tsv_path = os.path.join(LIAR_DIR, f"{stem}.tsv")
            if os.path.isfile(csv_path) and not os.path.isfile(tsv_path):
                os.rename(csv_path, tsv_path)
        return _tsv_exists()
    except Exception as e:
        print(f"    ❌ Kaggle failed: {e}")
        return False


def download_liar():
    if _tsv_exists():
        print("✅ LIAR already downloaded.")
        return

    print("⏳ Downloading LIAR dataset…")
    for attempt, fn in enumerate([_try_ucsb, _try_github, _try_kaggle], 1):
        try:
            if fn():
                print(f"✅ Downloaded via source {attempt}.")
                return
        except Exception as e:
            print(f"    Source {attempt} error: {e}")

    raise RuntimeError(
        "All download sources failed.\n"
        "Manual fix: go to https://www.kaggle.com/datasets/ucsbnlp/liar,\n"
        "download the zip, extract train.tsv / valid.tsv / test.tsv into:\n"
        f"  {LIAR_DIR}/"
    )


download_liar()

# Inspect the actual columns of the downloaded file
_sample = pd.read_csv(os.path.join(LIAR_DIR, "train.tsv"), sep="\t", header=None, nrows=2)
print(f"\nDetected {_sample.shape[1]} columns in train.tsv")
print(f"Files: {os.listdir(LIAR_DIR)}")


### A.2 Load & preprocess LIAR

In [ ]:

# 6-class LIAR label string → 3-class credibility int
LABEL_MAP = {
    "pants-fire": 0,   # False
    "false":      0,
    "barely-true": 1,  # Uncertain
    "half-true":  1,
    "mostly-true": 2,  # Credible
    "true":       2,
}

# Full 14-column schema (some downloads only have 13 — context column absent)
LIAR_COLUMNS_14 = [
    "id", "label", "statement", "subject", "speaker",
    "speaker_job", "state_info", "party",
    "barely_true_count", "false_count", "half_true_count",
    "mostly_true_count", "pants_fire_count", "context",
]
LIAR_COLUMNS_13 = LIAR_COLUMNS_14[:13]


def load_liar_split(filename):
    path = os.path.join(LIAR_DIR, filename)
    assert os.path.isfile(path), f"Missing: {path}"

    # Auto-detect column count from first row
    with open(path, "r", encoding="utf-8", errors="replace") as fh:
        first_line = fh.readline()
    n_cols = len(first_line.split("\t"))
    col_names = LIAR_COLUMNS_14 if n_cols >= 14 else LIAR_COLUMNS_13
    # Handle files with even more columns
    if n_cols > len(col_names):
        col_names = col_names + [f"extra_{i}" for i in range(n_cols - len(col_names))]

    df = pd.read_csv(
        path, sep="\t", header=None,
        names=col_names[:n_cols],        # only as many names as actual columns
        on_bad_lines="skip",
        quoting=3,                        # QUOTE_NONE — LIAR has unbalanced quotes
        encoding="utf-8",
        encoding_errors="replace",        # pandas >= 1.3 — replaces bad bytes instead of crashing
    )

    assert "label"     in df.columns, f"'label' column not found — check {filename}"
    assert "statement" in df.columns, f"'statement' column not found — check {filename}"

    df = df[["statement", "label"]].dropna()
    df["label_3"] = df["label"].str.strip().map(LABEL_MAP)
    df = df.dropna(subset=["label_3"])
    df["label_3"] = df["label_3"].astype(int)

    assert len(df) > 0, (
        f"⚠️  {filename} parsed to 0 rows!\n"
        f"  Detected {n_cols} columns, used names: {col_names[:n_cols]}\n"
        f"  Raw first line: {first_line[:200]}"
    )
    return df


train_df = load_liar_split("train.tsv")
val_df   = load_liar_split("valid.tsv")
test_df  = load_liar_split("test.tsv")

print(f"Train: {len(train_df):,}  |  Val: {len(val_df):,}  |  Test: {len(test_df):,}")
print(f"\nFirst training example:")
print(f"  label={train_df['label'].iloc[0]!r}  →  {train_df['label_3'].iloc[0]}")
print(f"  statement: {train_df['statement'].iloc[0][:100]}")
print("\nLabel distribution (train):")
print(train_df["label_3"].value_counts().sort_index().rename({0: "False", 1: "Uncertain", 2: "Credible"}))


### A.3 Tokenize & build DataLoaders

In [ ]:

# ── Tokenizer ─────────────────────────────────────────────────────────────────
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)
MAX_LEN   = 256

# ── PyTorch Dataset wrapper ───────────────────────────────────────────────────
class TextDataset(TorchDataset):
    def __init__(self, df: pd.DataFrame):
        """df must have columns: 'statement' (str) and 'label_3' (int 0/1/2)."""
        texts  = df["statement"].astype(str).tolist()
        labels = df["label_3"].tolist()
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=MAX_LEN,
            return_tensors="pt",
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

# ── Build datasets ────────────────────────────────────────────────────────────
print("⏳ Tokenizing LIAR splits…")
train_ds = TextDataset(train_df.reset_index(drop=True))
val_ds   = TextDataset(val_df.reset_index(drop=True))
test_ds  = TextDataset(test_df.reset_index(drop=True))

assert len(train_ds) > 0, "train_ds is empty — re-run the download + load cells"
assert len(val_ds)   > 0, "val_ds is empty — re-run the download + load cells"
print(f"✅ Dataset sizes — train: {len(train_ds)}, val: {len(val_ds)}, test: {len(test_ds)}")


### A.4 Train DistilBERT (global)

In [ ]:
model = DistilBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)

args = TrainingArguments(
    output_dir=MODEL_DIR,
    num_train_epochs=EPOCHS_GLOBAL,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
)

print("🚀 Stage A — Training DistilBERT on LIAR…")
trainer.train()

### A.5 Evaluate on LIAR test set

In [ ]:
results = trainer.evaluate(test_ds)
print("\n\ud83d\udcca LIAR Test-set results:")
for k, v in results.items():
    print(f"   {k}: {v:.4f}" if isinstance(v, float) else f"   {k}: {v}")

### A.6 Save global checkpoint

In [ ]:
trainer.save_model(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)
print(f"\u2705 Global model saved to {MODEL_DIR}")

---
## Summary & Next Steps

**What was trained:**
1. ✅ **DistilBERT** — trained on LIAR (12.8k labelled claims, 6-class → 3-class)

**Files produced:**
- `./models/liar_distilbert/` — text credibility classifier

**To deploy:**
1. Download `./models/` folder from Colab
2. Place in your Veritas project root
3. Run `python app.py`

**To improve accuracy further:**
- Add more labelled claims to a CSV and continue fine-tuning from the saved checkpoint
- Increase `EPOCHS_GLOBAL` or add a second fine-tuning pass with a lower learning rate